<a href="https://colab.research.google.com/github/Susan-sapkota-98/Nepali-asr-evaluation/blob/main/13_rate_test_set.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from pathlib import Path
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")
folder = Path("/content/drive/MyDrive/nepali-asr-evaluation/rating_200")


Mounted at /content/drive


In [3]:
rating_sheet = pd.read_csv(folder / "rating_sheet_test.csv", dtype=str, keep_default_na=False)
my_ratings_path = folder / "my_ratings_test.csv"

if my_ratings_path.exists():
    my_ratings = pd.read_csv(my_ratings_path, dtype=str, keep_default_na=False)
else:
    my_ratings = pd.DataFrame(columns=["file_id", "rating", "source"])

if "source" not in my_ratings.columns:
    my_ratings["source"] = "unknown"

print(len(rating_sheet), "rows in the sheet")
print(len(my_ratings), "rows already rated")

156 rows in the sheet
0 rows already rated


## The rule (version 3.1, readable)
I ask myself: if I read the model text, do I understand the same meaning as the correct text, without effort?
- 1 = yes. A space, short/long vowel, chandrabindu, punctuation, a small letter slip, or a spoken-style spelling of the same word
- 0 = no. A different word, a changed name, English letters, junk, a wrong number, or many letters wrong
- ? = I cannot decide
- q = stop for now (my answers are saved)

In [5]:
print("Question: same meaning, without effort?  1 = yes, 0 = no")
print("Only add c if the answer came from someone else, like 0c")
print()

for i in range(len(rating_sheet)):
    file_id = rating_sheet.loc[i, "file_id"]
    if file_id in list(my_ratings["file_id"]):
        continue

    print("row", i + 1, "of", len(rating_sheet))
    print("correct text:", rating_sheet.loc[i, "correct_text"])
    print("model text:  ", rating_sheet.loc[i, "model_text"])
    answer = input("1, 0, ?, q = stop: ")

    if answer == "q":
        break

    source = "me"
    if answer.endswith("c"):
        source = "claude"
        answer = answer[:-1]

    if answer not in ["1", "0", "?"]:
        print("wrong key, this row is skipped. Run this cell again to rate it.")
        continue

    my_ratings.loc[len(my_ratings)] = [file_id, answer, source]
    my_ratings.to_csv(my_ratings_path, index=False)
    print()

Question: same meaning, without effort?  1 = yes, 0 = no
Only add c if the answer came from someone else, like 0c

row 35 of 156
correct text: सन् १९७५ मा
model text:   सन् 1975 मा
1, 0, ?, q = stop: 1

row 36 of 156
correct text: इतिहास लेख्दैनन्
model text:   इतिहास लेग्देनन्
1, 0, ?, q = stop: ?

row 37 of 156
correct text: मात्र रहेको खलङ्गामा
model text:   मात्रो रहेको खलङ्गामा
1, 0, ?, q = stop: 0

row 38 of 156
correct text: धामी झाँक्री सबैमा
model text:   दामी झाँक्री सबैमा
1, 0, ?, q = stop: 0

row 39 of 156
correct text: सबैभन्दा सुन्दर र पवित्र
model text:   सबै भन्दा सुन्द र पवित्र
1, 0, ?, q = stop: 0

row 40 of 156
correct text: भन्न त सक्छु
model text:   भन्नत सक्छु
1, 0, ?, q = stop: 1

row 41 of 156
correct text: काँडे भ्याकुर समुद्र
model text:   काड्े भ्याकुर समुद्र
1, 0, ?, q = stop: ?

row 42 of 156
correct text: मुख्यदेखि बौद्ध
model text:   मुखे देखि बौद्ध
1, 0, ?, q = stop: 0

row 43 of 156
correct text: ड्रागनफल सिउँडी परिवारको
model text:   ड्रागन फल सिउडी पर

In [6]:
print(len(my_ratings), "rows rated of", len(rating_sheet))
print(my_ratings["rating"].value_counts())

156 rows rated of 156
rating
0    80
1    51
?    25
Name: count, dtype: int64


In [7]:
private = pd.read_csv(folder / "rating_rows_private_test.csv", dtype={"file_id": str}, keep_default_na=False)
joined = my_ratings.merge(private, on="file_id")

print("unsure rows:", (joined["rating"] == "?").sum())

rated = joined[joined["rating"].isin(["0", "1"])].copy()
rated["rating"] = rated["rating"].astype(int)
rated["forgiven"] = rated["folded_cer"] == 0
print("rated rows (1 or 0):", len(rated))

unsure rows: 25
rated rows (1 or 0): 131


In [8]:
pd.crosstab(rated["group"], rated["rating"])

rating,0,1
group,,
harmless,0,17
needs a human,61,34
real mistake,19,0


In [9]:
rated.groupby("rating")[["wer", "cer", "folded_cer"]].mean().round(2)

,wer,cer,folded_cer
rating,,,
0,0.73,0.31,0.28
1,0.55,0.10,0.02


In [10]:
for score_name in ["wer", "cer", "folded_cer"]:
    print(score_name, round(rated["rating"].corr(rated[score_name], method="spearman"), 2))

wer -0.16
cer -0.52
folded_cer -0.78


In [11]:
pd.crosstab(rated["rating"], rated["forgiven"])

forgiven,False,True
rating,,
0,80,0
1,18,33


In [12]:
accepted_left = rated[(rated["rating"] == 1) & (~rated["forgiven"])].sort_values("folded_cer", ascending=False)
print("accepted but still different:", len(accepted_left))
accepted_left[["correct_text", "model_text", "folded_cer"]].head(8)

accepted but still different: 18


,correct_text,model_text,folded_cer
97,२००५ को वृत्तचित्र,2005 को विद्दचित्र,0.214286
82,साथ अर्का गायक,सात अर्का गायक,0.090909
66,मध्यको एक अवतार,मध्येको एक अवतार,0.083333
145,ग्रामीण गाउँ तथा,ग्रामिन गाउँ तथा,0.076923
155,एकअन्तर्गत ११ वडामा,एक अन्तर्गत 11: ओडामा,0.066667
21,अभियान साबित भएको,अभियान सावित भएको,0.066667
107,सोही परिणामको कार्बन,सोही परिणामको कार्वन,0.058824
27,पाटी सञ्चालन गरेका थिए,पार्टी सञ्चालन गरेका थिए,0.055556


In [14]:
too_forgiving = rated[(rated["rating"] == 0) & (rated["forgiven"])]
print("rejected but forgiven:", len(too_forgiving))
too_forgiving[["correct_text", "model_text"]]

rejected but forgiven: 0


,correct_text,model_text


In [15]:
for fill_value, name in [("0", "all ? as 0"), ("1", "all ? as 1")]:
    all_rows = joined.copy()
    all_rows["rating"] = all_rows["rating"].replace("?", fill_value).astype(int)
    print(name)
    for score_name in ["wer", "cer", "folded_cer"]:
        print("  ", score_name, round(all_rows["rating"].corr(all_rows[score_name], method="spearman"), 2))

all ? as 0
   wer -0.17
   cer -0.47
   folded_cer -0.69
all ? as 1
   wer -0.07
   cer -0.42
   folded_cer -0.68
